## Kernel setup
To run this notebook, select the kernel named `.venv (Python 3.11.4)` from the kernel/interpreter picker in VS Code. If the kernel isn't listed, restart VS Code or run:

`python -m ipykernel install --user --name .venv --display-name ".venv (Python 3.11.4)"`


In [ ]:
# Verify kernel environment
import sys
print("Python:", sys.executable)
print("Version:", sys.version)


Perfect! The notebook kernel .venv is missing and imports show `ModuleNotFoundError: No module named 'ffmpeg'`. My next step is to configure the Python environment for the workspace so I can recreate .venv and install the required packages.

Configuring a Python Environment
Completed with input: {
  "resourcePath": "c:\\Users\\amitt\\VIDEO-AI-NLP-and-Deep-Learning-Applications"
}

Proceeding to create a .venv, install required packages (including `ffmpeg-python` and `ipykernel`), and register the .venv Jupyter kernel.

Ran terminal command: C:/Python311/python.exe -m venv .venv; .venv\Scripts\python.exe -m pip install --upgrade pip setuptools wheel; .venv\Scripts\python.exe -m pip install ipykernel ffmpeg-python numpy==1.26.4 soundfile opencv-python==4.9.0.80 openai-whisper "datasets[audio]" transformers accelerate; .venv\Scripts\python.exe -m pip install --index-url https://download.pytorch.org/whl/cpu torch torchvision torchaudio; .venv\Scripts\python.exe -m ipykernel install --user --name .venv --display-name ".venv (Python 3.11.4)"

Ran terminal command: .venv\Scripts\python.exe -c "import importlib, pkgutil; import ffmpeg; print('ffmpeg OK', ffmpeg.__version__)"; .venv\Scripts\python.exe -m pip show ipykernel ; echo no-ipykernel

Read [](file:///c%3A/Users/amitt/VIDEO-AI-NLP-and-Deep-Learning-Applications/Notebook%201.ipynb#1-1), lines 1 to 160

Ran terminal command: .venv\Scripts\python.exe -c "import importlib, sys
modules = ['ffmpeg','cv2','torch','whisper','transformers','datasets']
for m in modules:
    try:
        importlib.import_module(m)
        print(m + ' OK')
    except Exception as e:
        print(m + ' ERROR:', e)
print('Python:', sys.executable)"

I'm going to add a small Python check cell and a short Markdown note at the top of the notebook to confirm the kernel and instruct how to select the .venv kernel.

Edit [](file:///c%3A/Users/amitt/VIDEO-AI-NLP-and-Deep-Learning-Applications/Notebook%201.ipynb)

Edit [](file:///c%3A/Users/amitt/VIDEO-AI-NLP-and-Deep-Learning-Applications/Notebook%201.ipynb)

Retrieving Notebook summary.

Running [](vscode-notebook-cell:/c%3A/Users/amitt/VIDEO-AI-NLP-and-Deep-Learning-Applications/Notebook%201.ipynb#X14sZmlsZQ%3D%3D)

Made changes.

## Video -> Timestamped Transcripts

In [5]:
#source .venv/bin/activate
#python -m pip install ipykernel
#python -m ipykernel install --user --name .venv --display-name ".venv (Python 3.12)"
#python -m jupyter notebook


In [ ]:
#PIP INSTALL
# !pip install --upgrade pip setuptools wheel

# Core numeric stack
%pip install numpy==1.26.4

# PyTorch
# !pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# Audio / video
%pip install soundfile
%pip install ffmpeg-python
%pip install opencv-python==4.9.0.80

# Whisper (replace espnet)
%pip install torchcodec
%pip install openai-whisper

# NLP stack
# !pip install transformers==4.38.2 datasets[audio]==2.18.0 accelerate==0.27.2
%pip install -U transformers datasets[audio] accelerate


# The error message indicates that the system is missing the 'libGL.so.1' library, which is required by OpenCV for certain operations.
# This can be resolved by installing the necessary system package.
%pip install libgl1

print("============== ✅ INSTALLATION COMPLETE ===================")


In [1]:
#IMPORTS
#OS
import subprocess
from pathlib import Path

#Set environment variables BEFORE importing datasets/transformers
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"
os.environ["HF_DATASETS_AUDIO_BACKEND"] = "soundfile"

#For Transcript Extraction
import ffmpeg
import cv2
import torch
import whisper
from torchvision import transforms
import numpy as np
import soundfile as sf

#ML
# from espnet_model_zoo.downloader import ModelDownloader
# from espnet2.bin.asr_inference import Speech2Text
from datasets import load_dataset, Audio
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# List the files in the current directory to verify the environment
print("CUDA available:", torch.cuda.is_available())
print("============== ✅ ALL PACKAGES IMPORTED ===================")
# %pip uninstall -y torchcodec


c:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: False
============== ✅ ALL PACKAGES IMPORTED ===================


In [2]:
# =========================
# ENVIRONMENT
# =========================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Force CPU
os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"
os.environ["HF_DATASETS_AUDIO_BACKEND"] = "soundfile"

# =========================
# IMPORTS
# =========================
import ffmpeg
import torch
import numpy as np
import os

from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# =========================
# DEVICE SETUP
# =========================
device = "cpu"
print("CUDA available:", torch.cuda.is_available())

# =========================
# LOAD WHISPER MODEL
# =========================
model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=False,
    use_safetensors=True,
).to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
)

# =========================
# FUNCTION: EXTRACT AUDIO FROM VIDEO
# =========================
def extract_audio_from_video(video_path, target_sr=16000):
    """
    Extract audio from video file and return mono float32 numpy array.
    """
    out, _ = (
        ffmpeg.input(video_path)
        .output('pipe:', format='f32le', acodec='pcm_f32le', ac=1, ar=target_sr)
        .run(capture_stdout=True, capture_stderr=True)
    )
    audio = np.frombuffer(out, dtype=np.float32)
    return audio, target_sr

# =========================
# FUNCTION: TRANSCRIBE AUDIO
# =========================
def transcribe_audio(audio_array, sampling_rate, language="english"):
    """
    Transcribe numpy audio array using Whisper pipeline.
    Returns a dict with 'text' and timestamped segments.
    """
    result = pipe(
        {"array": audio_array, "sampling_rate": sampling_rate},
        generate_kwargs={"language": language},
        return_timestamps=True,
    )
    return result

# =========================
# FUNCTION: PROCESS SINGLE VIDEO
# =========================
def transcribe_video_file(video_path, language="english"):
    print(f"Processing {video_path} ...")
    
    # Extract audio
    audio_array, sampling_rate = extract_audio_from_video(video_path)
    
    # Transcribe
    result = transcribe_audio(audio_array, sampling_rate, language=language)
    
    # Save transcript
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    transcript_path = os.path.join(os.path.dirname(video_path), base_name + "-transcript.txt")
    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write(result["text"])
    
    print(f"✅ Transcript saved: {transcript_path}")
    return result

# =========================
# BATCH PROCESS: DIRECTORY
# =========================
directory = "Training Data"  # replace with your folder path

for file in os.listdir(directory):
    if file.lower().endswith((".mp4", ".mov")):
        video_path = os.path.join(directory, file)
        transcribe_video_file(video_path)


CUDA available: False


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 587/587 [05:10<00:00,  1.89it/s, Materializing param=model.encoder.layers.31.self_attn_layer_norm.weight] 


Processing Training Data\Week 01 - Embedded S.mp4 ...


AttributeError: module 'ffmpeg' has no attribute 'input'

In [ ]:
# =========================
# ENVIRONMENT
# =========================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Force CPU
os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"
os.environ["HF_DATASETS_AUDIO_BACKEND"] = "soundfile"

# =========================
# IMPORTS
# =========================
import ffmpeg
import torch
import numpy as np
import os

from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# =========================
# DEVICE SETUP
# =========================
device = "cpu"
print("CUDA available:", torch.cuda.is_available())

# =========================
# LOAD WHISPER MODEL
# =========================
model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=False,
    use_safetensors=True,
).to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
)

# =========================
# FUNCTION: EXTRACT AUDIO FROM VIDEO
# =========================
def extract_audio_from_video(video_path, target_sr=16000):
    """
    Extract audio from video file and return mono float32 numpy array.
    """
    out, _ = (
        ffmpeg.input(video_path)
        .output('pipe:', format='f32le', acodec='pcm_f32le', ac=1, ar=target_sr)
        .run(capture_stdout=True, capture_stderr=True)
    )
    audio = np.frombuffer(out, dtype=np.float32)
    return audio, target_sr

# =========================
# FUNCTION: SPLIT AUDIO INTO CHUNKS
# =========================
def chunk_audio(audio_array, sampling_rate, chunk_length_sec=30):
    """
    Split audio into chunks of chunk_length_sec.
    Returns list of (start_time_sec, audio_chunk) tuples.
    """
    chunk_size = chunk_length_sec * sampling_rate
    chunks = []
    for i in range(0, len(audio_array), chunk_size):
        chunk = audio_array[i:i + chunk_size]
        start_time = i / sampling_rate
        chunks.append((start_time, chunk))
    return chunks

# =========================
# FUNCTION: TRANSCRIBE CHUNKS
# =========================
def transcribe_audio_chunks(chunks, sampling_rate, language="english"):
    """
    Transcribe list of audio chunks. Returns concatenated text with timestamps.
    """
    full_transcript = []
    for start_time, chunk in chunks:
        if len(chunk) == 0:
            continue
        result = pipe(
            {"array": chunk, "sampling_rate": sampling_rate},
            generate_kwargs={"language": language},
            return_timestamps=True,
        )
        for segment in result.get("chunks", []):
            # Adjust timestamps relative to original audio
            segment["start"] += start_time
            segment["end"] += start_time
            full_transcript.append(segment)
    return full_transcript

# =========================
# FUNCTION: SAVE TRANSCRIPT TO FILE
# =========================
def save_transcript(transcript_segments, video_path):
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    transcript_path = os.path.join(os.path.dirname(video_path), base_name + "-transcript.txt")

    with open(transcript_path, "w", encoding="utf-8") as f:
        for seg in transcript_segments:
            start = seg["start"]
            end = seg["end"]
            text = seg["text"]
            f.write(f"[{start:.2f}s - {end:.2f}s] {text}\n")

    print(f"✅ Transcript saved: {transcript_path}")

# =========================
# FUNCTION: PROCESS SINGLE VIDEO
# =========================
def transcribe_video_file(video_path, language="english", chunk_length_sec=30):
    print(f"Processing {video_path} ...")
    
    # Extract audio
    audio_array, sampling_rate = extract_audio_from_video(video_path)
    
    # Split into chunks
    chunks = chunk_audio(audio_array, sampling_rate, chunk_length_sec=chunk_length_sec)
    
    # Transcribe all chunks
    transcript_segments = transcribe_audio_chunks(chunks, sampling_rate, language=language)
    
    # Save transcript
    save_transcript(transcript_segments, video_path)

# =========================
# BATCH PROCESS DIRECTORY
# =========================
directory = "Training Data"  # replace with your folder path

for file in os.listdir(directory):
    if file.lower().endswith((".mp4", ".mov")):
        video_path = os.path.join(directory, file)
        transcribe_video_file(video_path, chunk_length_sec=30)


In [ ]:
# =========================
# LOAD PRE-TRAINED AVSR MODEL (OPENAI WHISPER MODEL)
# =========================

# =========================
# ENVIRONMENT (MUST BE FIRST)
# =========================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"
os.environ["HF_DATASETS_AUDIO_BACKEND"] = "soundfile"

# =========================
# IMPORTS
# =========================
import io
import torch
import soundfile as sf
import numpy as np

from datasets import load_dataset, Audio
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

print("CUDA available:", torch.cuda.is_available())

# =========================
# LOAD WHISPER MODEL
# =========================
device = "cpu"
model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=False,
    use_safetensors=True,
).to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
)

# =========================
# LOAD DATASET (NO AUDIO DECODING)
# =========================
dataset = load_dataset(
    "distil-whisper/librispeech_long",
    "clean",
    split="validation",
)

dataset = dataset.cast_column(
    "audio",
    Audio(decode=False),
)

# =========================
# MANUAL AUDIO LOADING (CRITICAL)
# =========================
audio_bytes = dataset[0]["audio"]["bytes"]   # <-- SAFE: path only
audio_array, sampling_rate = sf.read(io.BytesIO(audio_bytes))

# Ensure mono float32
if audio_array.ndim > 1:
    audio_array = np.mean(audio_array, axis=1)
audio_array = audio_array.astype(np.float32)

# =========================
# TRANSCRIBE
# =========================
result = pipe(
    {"array": audio_array, "sampling_rate": sampling_rate},
    generate_kwargs={"language": "english"},
    return_timestamps=True,
)


print(result["text"])


In [ ]:
# =========================
# LOAD TRAINING DATA (VIDEO & TRANSCRIPTS)
# =========================

# TODO: CONVERT THE CODE INTO FUNCTIONS
#Train on whole directory of video and text files

directory = "Training Data"
os.listdir(directory)
text_files = {}
print("Raw Data Directory: Transcripts (Data)")
print("====================================================================================")

    
count = 1

for file in os.listdir(directory):
    file_path = os.path.join(directory, file)

    # Skip directories
    if not os.path.isfile(file_path):
        continue

    print(file_path, "has size", os.path.getsize(file_path), "bytes.")

    # Process video files only
    if file.lower().endswith((".mp4", ".mov")):
        base_name = os.path.splitext(file)[0]
        transcript_name = base_name + "-transcript.txt"
        transcript_path = os.path.join(directory, transcript_name)

        # If transcript does not exist → generate it
        if not os.path.exists(transcript_path):
            print(f"⚠️ Transcript missing for {file}")
            continue

            # audio_path = os.path.join(directory, base_name + ".wav")
            # extract_audio_from_video(file_path, audio_path)

            # audio = load_audio(audio_path)
            # transcript = speech2text(audio)

            # with open(transcript_path, "w", encoding="utf-8") as f:
            #     f.write(transcript)

            # print(f"✅ Transcript generated: {transcript_path}")

        # Load transcript (existing or newly created)
        with open(transcript_path, "r", encoding="utf-8") as f:
            text_files[f"Transcript_{count}"] = f.read()
            print(f"✅ Loaded: {transcript_path} successfully!")
            count += 1    


#uploads one file at a time
# from tkinter import Tk
# from tkinter.filedialog import askopenfilename

# Tk().withdraw()  # Hide GUI window

# file_path = askopenfilename(filetypes=["*.txt"])

# with open(file_path, "r", encoding="utf-8") as file:
#     text = file.read()

# print(f"Uploaded file: {file_path}")


## Pre-processing the Transcripts (.txt files)

In [ ]:
#Function to clean the uploaded text file
import re
import string
%pip install -U nltk
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

def clean_transcript(filename):

    # Read the text file
    with open(filename, 'r') as file:
        text = file.read()
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Tokenize the text
    tokens = word_tokenize(text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    # Lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    # Join the tokens back into a single string
    cleaned_text = ' '.join(tokens)
    # Save cleaned textfile in specific directory
    with open('cleaned_text.txt', 'w') as file:
        file.write(cleaned_text)
    return cleaned_text


In [ ]:
#NLP model - tasks that can be achieved

# 1. Word Frequency

from collections import Counter

word_freq = Counter(tokens)
print(word_freq.most_common(10))

# 2. 🔹 Named Entity Recognition (NER)

for ent in doc.ents:
    print(ent.text, ent.label_)

# 3. 🔹 Sentiment analysis (example with TextBlob)

from textblob import TextBlob

blob = TextBlob(text)
print(blob.sentiment)





Sliding Window Version - 1

In [ ]:
def sliding_window_keyframe_extraction(frames: list[np.ndarray], window_size: 12, step_size: int, threshold: 0.5) -> list[np.ndarray]:
    key_frames = []
    num_frames = len(frames)

    # Compute sum of first window of size = window_size
    delta = cosine_similarity(frames[0], frames[window_size -1])
    if delta > -threshold and delta < threshold:
        key_frames.append(frames[window_size -1])


    for i in range (1, num_frames - window_size + 1, step_size):
        # Sliding window: remove first frame and add next one
        end_frame = frames[i+ window_size - 1]
        delta = cosine_similarity(frames[i], end_frame)
        # delta = np.linalg.norm(frames[i] - end_frame)
        # delta = delta - frames[i - 1] + frames[i + window_size - 1]
        if delta > -threshold and delta < threshold:
            key_frames.append([frames[i], end_frame])

     
    # for start in range(0, num_frames - window_size + 1, step_size):
    #     window = frames[start:start + window_size]
    #     # Compute differences between consecutive frames in the window
    #     diffs = [torch.sum(torch.abs(window[i] - window[i-1])).item() for i in range(1, len(window))]
    #     avg_diff = sum(diffs) / len(diffs)
        
    #     # Select the frame with the maximum difference as key frame
    #     max_diff_index = diffs.index(max(diffs)) + 1  # +1 to account for offset
    #     key_frames.append(window[max_diff_index])
    
    return key_frames


boundary_frames = sliding_window_keyframe_extraction(video_data['pixel_values'][0], step_size = 16)
print(f"Number of key frames: {len(boundary_frames)}\n")

for i in boundary_frames: 
    print(i.dtype, i.shape)

"""
Return embeddings from the key frames and map with respective timestamps & text:
{
    start_time: datetime,
    end_time: datetime,
    transcript: str,
    text_embedding: np.ndarray,
    visual_embedding: np.ndarray
}
"""
def map_keyframes_to_transcript(key_frames: List[np.ndarray], extracted_segments: List[str, Tuple[datetime, datetime, str]], np.ndarray, np.ndarray) -> List[dict]:
    mapped_keyframes = []
    for i in key_frames:
        fps = video_data['avg_fps']
        frame_start_time = i[0]/fps
        frame_end_time = i[1]/fps
        transcript = ""
        for j in extracted_segments[1]:
            start_time = j[0].hour*3600 + j[0].minute*60 + j[0].second*60 + j[0].microseconds/1e6
            end_time = j[1].hour*3600 + j[1].minute*60 + j[1].second*60 + j[1].microseconds/1e6
            if j[0] >= frame_start_time and j[1] =< frame_end_time:
                transcript += j[2]
                
        mapping = {
            "start_time": frame_start_time,
            "end_time": frame_end_time,
            "transcript": transcript
        }

        mapped_keyframes.append(mapping)
                
    return mapped_keyframes

mapped_keywords = map_keyframes_to_transcript(boundary_frames, EXTRACTED_SEGMENTS)

for i in mapped_keywords:
    print(i)


Visual Embedding Numpy Error prone code

In [ ]:
# CHATGPT IMPROVED CODE
"""
FUNCTION: Detect semantic shifts in the video using TimeSformer model
INPUT: np.ndarray: textual_features: np.ndarray of shape (N, D)
       float: threshold - Confidence threshold for detecting shifts
OUTPUT: list[int]: Indices where semantic shifts are detected
"""

def detect_semantic_shifts_in_video(
    visual_features: np.ndarray,
    threshold: float = 0.3
) -> list[int]:

    model = TimesformerModel.from_pretrained(
        "facebook/timesformer-base-finetuned-k400"
    )
    model.eval()

    # Detect shifts via cosine similarity
    shift_indices = []
    for index in range(1, len(visual_features), 1):
        similarity_check = cosine_similarity(visual_features[index], visual_features[index - 1])
        if similarity_check < threshold:
            shift_indices.append(index)

    return shift_indices


"""
FUNCTION: Detect potential in-video chapters/indexes based on semantic shifts
INPUT: list[int]: shift_indices - Frame indices where shifts occur
       dict: video_frames - Video metadata including duration and total_frames
OUTPUT: list[tuple[float, float]]: Time ranges (start, end) of detected chapters
"""

def detect_chapters_in_video(shift_indices: list[int], video_data: dict) -> list[tuple[float, float]]:
    chapters = []
    frame_duration = video_data['duration'] / video_data['total_frames']

    # Add start boundary
    boundaries = [0] + shift_indices + [video_data['total_frames'] - 1]
    
    if len(shift_indices) < 2:
        print("WARNING: Extracted Segments not available or Semantic Shift indices detected were < 2.")
    
    for i in range(0, len(boundaries)-1, 1):
        start_time = boundaries[shift_indices[i]][0]
        end_time = boundaries[shift_indices[i+1]][0]

        # chapters.append(extracted_segments[index])
        chapters.append((start_time * frame_duration, end_time * frame_duration))

    return chapters

"""
FUNCTION: Detect similarities between two video segments using cosine similarity
INPUT: np.ndarray: vec1 - First feature vector
       np.ndarray: vec2 - Second feature vector
OUTPUT: float: Cosine similarity value between 0 and 1
"""

def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    
    # Handle zero-magnitude vectors
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    
    return float(dot_product / (norm_vec1 * norm_vec2))

def extract_timesformer_embeddings(video_data: dict) -> np.ndarray:
    """
    Returns frame/clip-level visual embeddings using TimeSformer CLS token
    Output shape: (T, 768)
    """

    model = TimesformerModel.from_pretrained(
        "facebook/timesformer-base-finetuned-k400"
    )
    model.eval()
    pixel_values = video_data["pixel_values"]

    with torch.no_grad():
        outputs = model(pixel_values.float())
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # (1, 768)

    return cls_embedding.squeeze(0).cpu().numpy()


# Execute detection pipeline
shifts, chapters, visual_embeddings = None, None, None
try:
    visual_embeddings = extract_timesformer_embeddings(video_data)

    # High similarity
    shifts = detect_semantic_shifts_in_video(visual_embeddings, threshold=0.3)
    print(f"Detected shift indices: {shifts}")
    # Low similarity
    # print(f"Detected low similarity shift indices: {detect_semantic_shifts_in_video(textual_features, threshold=0.3)}")
    
    chapters = detect_chapters_in_video(video_data, shifts)
    print(f"In-Video Chapters:\n{chapters}")
    

except Exception as e:
    print(f"Error during semantic shift detection: {type(e).__name__}: {e}")
    print("Ensure video_data and textual_features are properly loaded from previous cells.")


Loading weights: 100%|██████████| 247/247 [00:00<00:00, 397.75it/s, Materializing param=layernorm.weight]                                        
[1mTimesformerModel LOAD REPORT[0m from: facebook/timesformer-base-finetuned-k400
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

[3mNotes:
- UNEXPECTED[3m	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.[0m
Error during semantic shift detection: RuntimeError: Given groups=1, weight of size [768, 3, 16, 16], expected input[3, 30, 224, 224] to have 3 channels, but got 30 channels instead
Ensure video_data and textual_features are properly loaded from previous cells.